# 252 — Clustering Recon Visualization (3D MNE Brain)

For each clustering CSV (e.g. `outputs/clustering/kmeans/labels_kmeans_raw_*.csv`):
- join each row with **fsaverage MNI coords** (`outputs/250_recon/fsaverage/coords/ALL_PATIENTS_contacts_fsaverage*.csv`)
- for each cluster, render the **fsaverage pial surface** (translucent) and overlay each electrode as a 3D sphere colored by group:
    - `cluster_<id>/by_patient/<view>.png` — colored by patient
    - `cluster_<id>/by_condition/<view>.png` — colored by condition
- save the merged-with-coords CSV and `UNMATCHED_contacts.csv` for QC

Views saved per cluster: `lateral_L`, `lateral_R`, `dorsal`, `frontal`.

Outputs go to: `outputs/250_recon/clustering_recon/<csv_basename>/cluster_<id>/<by_*>/`

**Toggles** (Cell A): `KEEP_WM`, `ALGOS`, `ALPHA`, `SPHERE_SCALE`, `SURF`, `CORTEX`.

**Contact-name join rule**
- **EL / BERN** (`EL030_audio_WM_ERSP_A_L10_TN.npy`) → contact `AL10`
- **PAT / GVA** (`PAT_3066_audio_WM_ERSP_AG10_TN.npy`) → contact `AG10`

> Uses **MNE-Python** + **PyVista** for 3D rendering (off-screen). Requires `mne`, `pyvista`, `pyvistaqt`.

In [9]:
# === Cell A: setup, paths, config (nilearn + MNE 3D, off-screen) ===
import os, sys, re, json, glob

# Process every run listed in outputs/clustering/index.json. Optionally filter
# to a subset by setting RUN_FILTER (list of dicts with any of: method,
# feature_set, run_id; matching is "all keys equal").
#
# Examples:
#   RUN_FILTER = None                                              # all runs
#   RUN_FILTER = [{"method": "kmeans"}]                            # all kmeans
#   RUN_FILTER = [{"method": "kmeans", "feature_set": "raw"}]      # one combo
#   RUN_FILTER = [{"run_id": "20260504_222635"}]                   # one run
RUN_FILTER = None
SKIP_IF_RECON_EXISTS = True   # don't re-render if <run_dir>/recon already populated

# Off-screen env BEFORE any pyvista/mne import
os.environ['PYVISTA_OFF_SCREEN']      = 'true'
os.environ['MNE_3D_OPTION_ANTIALIAS'] = 'true'
os.environ['MESA_GL_VERSION_OVERRIDE'] = '3.3'

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import plotting as nlp

# ---- repo paths ----
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
FUNCTIONS_DIR = NOTEBOOK_DIR / 'functions'
if str(FUNCTIONS_DIR) not in sys.path:
    sys.path.insert(0, str(FUNCTIONS_DIR))
import lf_recon_shared_config as C

# ---- I/O roots ----
OUT_ROOT          = Path(C.OUTPUTS_ROOT)
print(OUT_ROOT)
FSAV_COORDS_DIR   = OUT_ROOT / '250_recon' / 'fsaverage' / 'coords'
CLUSTERING_DIR    = OUT_ROOT / 'clustering'                # manifest-driven home
print(CLUSTERING_DIR)
CLUSTERING_INDEX  = CLUSTERING_DIR / 'index.json'
run_dir = CLUSTERING_DIR
# Recon images are written INSIDE each run dir at <run_dir>/recon/ — that's the
# path results.html's loadClusterRecon tries first, so the website finds them
# automatically without any URL gymnastics.

# ---- toggles ----
KEEP_WM      = False
ALGOS        = None
DPI          = 350

# ---- MNE 3D Brain config ----
SUBJECTS_DIR = Path('//nasac-m2.unige.ch/m-HumanNeuronLab/DATARAW/SEEG_EXPERIMENTS_BERN/Reconstruction')
SUBJECT      = 'fsaverage'
SURF         = 'pial'
CORTEX       = 'bone'
ALPHA        = 0.2
BACKGROUND   = 'white'
BRAIN_SIZE   = (1200, 1000)   # window size for off-screen render
SPHERE_SCALE = 0.3            # mm (smaller = smaller dots; tweak here)

VIEWS = {
    'lateral_L': dict(view='lateral', hemi='lh'),
    'lateral_R': dict(view='lateral', hemi='rh'),
    'dorsal'   : dict(view='dorsal',  hemi='both'),
    'frontal'  : dict(view='frontal', hemi='both'),
}

# ---- pyvista off-screen + MNE pyvista backend (no Qt) ----
import pyvista as pv
pv.OFF_SCREEN = True
# Set the GLOBAL theme window_size before creating the Brain — this is the
# size the off-screen render window will be created at on this pyvista version.
pv.global_theme.window_size            = list(BRAIN_SIZE)
pv.global_theme.background             = BACKGROUND
pv.global_theme.transparent_background = True
pv.global_theme.full_screen            = False
pv.global_theme.anti_aliasing          = 'msaa'

import mne
try:
    mne.viz.set_3d_backend('pyvista')
except Exception:
    mne.viz.set_3d_backend('notebook')
if hasattr(mne.viz, 'set_3d_options'):
    try:
        mne.viz.set_3d_options(antialias=True, depth_peeling=True, smooth_shading=True)
    except Exception:
        pass
os.environ['SUBJECTS_DIR'] = str(SUBJECTS_DIR)
print('mne     :', mne.__version__)
print('pyvista :', pv.__version__)
print('3D backend :', mne.viz.get_3d_backend())
print('global_theme.window_size :', pv.global_theme.window_size)

# sanity-check fsaverage
fsav_subj = SUBJECTS_DIR / SUBJECT
fsav_surf = fsav_subj / 'surf'
assert fsav_surf.exists() and (fsav_surf/'lh.pial').exists() and (fsav_surf/'rh.pial').exists(), \
    f'fsaverage surface files missing in {fsav_surf!r}'
print('fsaverage surf OK')

# ---- load fsaverage coords ----
fsav_csv = FSAV_COORDS_DIR / ('ALL_PATIENTS_contacts_fsaverage.csv' if KEEP_WM
                              else 'ALL_PATIENTS_contacts_fsaverage_nowm.csv')
assert fsav_csv.exists(), f'Missing {fsav_csv}'
fsav = pd.read_csv(fsav_csv)
print(f'\nLoaded fsaverage coords: {len(fsav)} contacts from {fsav_csv.name}')
fsav.head()

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs
\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering
mne     : 1.6.1
pyvista : 0.46.3
3D backend : pyvistaqt
global_theme.window_size : [1200, 1000]
fsaverage surf OK

Loaded fsaverage coords: 2838 contacts from ALL_PATIENTS_contacts_fsaverage_nowm.csv


,patient,cohort,name,name_raw,hemi,x,y,z,is_wm
0,EL030,BERN,AL1,AL1 D L,L,-23.591169,-7.959251,-18.232117,0
1,EL030,BERN,AL2,AL2 D L,L,-25.317326,-4.529516,-21.213720,0
2,EL030,BERN,AL3,AL3 D L,L,-26.150476,-3.044088,-21.015188,0
3,EL030,BERN,AL4,AL4 D L,L,-32.918415,3.394147,-22.629436,0
4,EL030,BERN,AL5,AL5 D L,L,-34.846104,-11.320873,-21.402191,0


In [10]:
# === Cell B: discover runs from index.json + define contact-name rule ===

def _basename_no_ext(p):
    p = str(p).replace('\\', '/')
    return os.path.basename(p).rsplit('.', 1)[0]

# BERN/EL  : <pat>_<cond>_WM_ERSP_<electrode>_<L|R><num>_TN... -> contact = electrode + side + num  (e.g. AL1, AR1)
# GVA/PAT  : <pat>_<cond>_WM_ERSP_<contact>_TN...               -> contact = electrode column directly
_RX_BERN = re.compile(r'_ERSP_([A-Za-z]+)_([LR])(\d+)_TN', re.IGNORECASE)
def _norm_contact(s):
    """Strip _, - and uppercase. Matches lf_io_utils.normalize_label so the
    labels side and the fsaverage side join cleanly regardless of how the
    contact was originally written (e.g. 'aH_R-1' / 'aH_R1' / 'AHR1')."""
    if s is None:
        return None
    return str(s).replace('_', '').replace('-', '').upper()

def contact_from_row(patient_id, electrode, file_path):
    base = _basename_no_ext(file_path)
    m = _RX_BERN.search(base)
    if m:
        return _norm_contact(f'{m.group(1)}{m.group(2)}{m.group(3)}')
    if isinstance(electrode, str) and electrode.strip():
        return _norm_contact(electrode)
    return None

# ---- read the manifest index ----
assert CLUSTERING_INDEX.exists(), f'Missing {CLUSTERING_INDEX}. Run 210 (or any clustering notebook) first.'
with open(CLUSTERING_INDEX) as f:
    _idx = json.load(f)

def _matches(run, flt):
    return all(run.get(k) == v for k, v in flt.items())

def _filter_runs(runs, filters):
    if not filters:
        return runs
    return [r for r in runs if any(_matches(r, f) for f in filters)]

runs_to_process = []
for r in _filter_runs(_idx.get('runs', []), RUN_FILTER):
    print(run_dir)
    run_dir   = CLUSTERING_DIR / r['path']
    labels_csv = run_dir / 'labels.csv'
    manifest_p = run_dir / 'manifest.json'
    recon_dir = run_dir / 'recon'
    if not labels_csv.exists():
        print(f'  [skip] {r["path"]} — labels.csv missing'); continue
    if not manifest_p.exists():
        print(f'  [skip] {r["path"]} — manifest.json missing'); continue
    if SKIP_IF_RECON_EXISTS and recon_dir.exists() and any(recon_dir.iterdir()):
        print(f'  [skip] {r["path"]} — recon already populated (set SKIP_IF_RECON_EXISTS=False to re-render)'); continue
    with open(manifest_p) as mf:
        manifest = json.load(mf)
    runs_to_process.append({
        'index_entry': r,
        'manifest'   : manifest,
        'labels_csv' : labels_csv,
        'run_dir'    : run_dir,
        'recon_dir'  : recon_dir,
    })

print(f'\nTotal runs to process: {len(runs_to_process)}')
for r in runs_to_process:
    m = r['manifest']
    print(f"  [{m['method']}/{m['feature_set']}] {m['run_id']}  -> {r['recon_dir']}")

# ---- load fsaverage coords ----
fsav_csv = FSAV_COORDS_DIR / ('ALL_PATIENTS_contacts_fsaverage.csv' if KEEP_WM
                              else 'ALL_PATIENTS_contacts_fsaverage_nowm.csv')
assert fsav_csv.exists(), f'Missing {fsav_csv}'
fsav = pd.read_csv(fsav_csv)
print(f'\nLoaded fsaverage coords: {len(fsav)} contacts from {fsav_csv.name}')


\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering
  [skip] hierarchical/blob/runs/20260521_163124 — labels.csv missing
\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\hierarchical\blob\runs\20260521_163124
  [skip] hierarchical/blob/runs/20260522_205440 — labels.csv missing
\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\hierarchical\blob\runs\20260522_205440
  [skip] hierarchical/blob/runs/20260522_213636 — labels.csv missing
\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\hierarchical\blob\runs\20260522_213636
  [skip] hierarchical/blob/runs/20260523_102742 — recon already populated (set SKIP_IF_RECON_EXISTS=False to re-render)
\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\hierarchical\blob\runs\202

In [11]:
# === Cell C: 3D translucent fsaverage brain + electrode spheres per cluster ===
from mne.viz import Brain

import matplotlib.colors as mcolors

def _safe_to_rgba(name, fallback=(0.5, 0.5, 0.5, 1.0)):
    """matplotlib.colors.to_rgba but with graceful fallback for unknown names (e.g. 'babyblue')."""
    try:
        return mcolors.to_rgba(name)
    except (ValueError, KeyError):
        return fallback

def _palette_patients(patient_ids):
    """Cohort-aware palette: EL patients use C.EL_COLOR_NAMES, PAT patients use C.PAT_COLOR_NAMES."""
    el_names  = list(getattr(C, 'EL_COLOR_NAMES',  []))
    pat_names = list(getattr(C, 'PAT_COLOR_NAMES', []))
    el_pats  = sorted({p for p in patient_ids if str(p).upper().startswith('EL')})
    pat_pats = sorted({p for p in patient_ids if str(p).upper().startswith('PAT')})
    other    = sorted({p for p in patient_ids if p not in el_pats and p not in pat_pats})
    palette  = {}
    for i, p in enumerate(el_pats):
        nm = el_names[i % len(el_names)] if el_names else 'tab:blue'
        palette[p] = _safe_to_rgba(nm)
    for i, p in enumerate(pat_pats):
        nm = pat_names[i % len(pat_names)] if pat_names else 'tab:red'
        palette[p] = _safe_to_rgba(nm)
    for i, p in enumerate(other):
        cmap = plt.get_cmap('tab10')
        palette[p] = cmap(i % 10)
    uniq = el_pats + pat_pats + other
    return palette, uniq

def _palette(values, cmap_name='tab10'):
    """Generic palette for non-patient grouping (e.g. condition)."""
    uniq = sorted(set(values))
    cmap = plt.get_cmap(cmap_name if len(uniq) <= 10 else 'tab20')
    n = max(len(uniq), 1)
    return {v: cmap(i / max(n-1, 1)) for i, v in enumerate(uniq)}, uniq

def _new_brain():
    try:
        return Brain(
            subject=SUBJECT, subjects_dir=str(SUBJECTS_DIR),
            surf=SURF, hemi='both',
            cortex=CORTEX, alpha=ALPHA,
            background=BACKGROUND, size=BRAIN_SIZE,
            offscreen=True, show=False, block=False,
        )
    except TypeError:
        return Brain(
            subject=SUBJECT, subjects_dir=str(SUBJECTS_DIR),
            surf=SURF, hemi='both',
            cortex=CORTEX, alpha=ALPHA,
            background=BACKGROUND, size=BRAIN_SIZE,
        )

def _force_size_and_render(brain):
    """Force the underlying pyvista plotter to honor BRAIN_SIZE and refresh GL."""
    try:
        plotter = brain._renderer.plotter
        plotter.window_size = list(BRAIN_SIZE)
        try:
            plotter.ren_win.SetSize(*BRAIN_SIZE)
        except Exception:
            pass
        plotter.reset_camera()
        plotter.render()
        return plotter
    except Exception as e:
        print(f'    [warn] could not resize plotter: {e}')
        return None

def _add_spheres(brain, coords_xyz_mm, colors_rgba, scale=SPHERE_SCALE):
    coords = np.asarray(coords_xyz_mm, dtype=float)
    for (xyz, c) in zip(coords, colors_rgba):
        hemi = 'lh' if xyz[0] < 0 else 'rh'
        try:
            brain.add_foci(
                xyz.reshape(1, 3),
                coords_as_verts=False,
                hemi=hemi,
                color=tuple(c[:3]),
                scale_factor=scale,
                alpha=1.0,
            )
        except Exception as e:
            print(f'    [warn] add_foci failed: {e}')

def _save_views(brain, out_dir, fname_prefix=''):
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {}
    for tag, kw in VIEWS.items():
        try:
            if kw['hemi'] in ('lh', 'rh'):
                brain.show_view(view=kw['view'], hemi=kw['hemi'])
            else:
                brain.show_view(view=kw['view'])
        except TypeError:
            brain.show_view(view=kw['view'])
        plotter = _force_size_and_render(brain)
        out_png = out_dir / f'{fname_prefix}{tag}.png'
        try:
            # Use pyvista plotter.screenshot directly with explicit window_size — most reliable
            if plotter is not None:
                plotter.screenshot(filename=str(out_png),
                                   transparent_background=False,
                                   window_size=list(BRAIN_SIZE))
            else:
                brain.save_image(str(out_png))
        except Exception as e1:
            try:
                brain.save_image(str(out_png))
            except Exception as e2:
                print(f'    [warn] save failed for {tag}: {e1} / {e2}')
                continue
        paths[tag] = out_png
    return paths

def _save_legend(palette, uniq, title, out_png):
    fig, ax = plt.subplots(figsize=(4, max(2, 0.25*len(uniq) + 1)))
    ax.axis('off')
    handles = [plt.Line2D([0],[0], marker='o', linestyle='', color=palette[v],
                          label=str(v), markersize=10) for v in uniq]
    ax.legend(handles=handles, loc='center', frameon=False, fontsize=10,
              ncol=1 if len(uniq) <= 18 else 2, title=title)
    fig.tight_layout()
    fig.savefig(out_png, dpi=DPI, bbox_inches='tight')
    plt.close(fig)

summary_rows = []

for r in runs_to_process:
    m          = r['manifest']
    csv_path   = r['labels_csv']
    out_dir    = r['recon_dir']
    out_dir.mkdir(parents=True, exist_ok=True)
    algo_tag   = f"{m['method']}_{m['feature_set']}"
    print(f"\n=== [{algo_tag}] {m['run_id']} -> {out_dir} ===")

    df = pd.read_csv(csv_path)
    cluster_col = f"cluster_{m['method']}_{m['feature_set']}"
    if cluster_col not in df.columns:
        # Fall back: pick the first column starting with `cluster_`
        cluster_cols = [c for c in df.columns if c.startswith('cluster_')]
        if not cluster_cols:
            print(f'  [skip] no cluster column (looked for {cluster_col})'); continue
        cluster_col = cluster_cols[0]

    if not {'patient_id', 'electrode', 'file_path', 'condition'}.issubset(df.columns):
        print(f'  [skip] labels.csv missing required metadata columns'); continue

    df['contact_name'] = df.apply(
        lambda row: contact_from_row(row['patient_id'], row['electrode'], row['file_path']),
        axis=1)
    # Normalize BOTH sides identically: strip _ and -, uppercase. This is
    # the same normalize_label rule used elsewhere; fixes the previous ~28%
    # UNMATCHED that came from raw names like 'aH_R-1' vs fsav 'AHR1'.
    df['contact_name'] = df['contact_name'].astype(str).map(_norm_contact)

    _fsav_join = fsav.copy()
    _fsav_join['name'] = _fsav_join['name'].astype(str).map(_norm_contact)
    merged = df.merge(
        _fsav_join[['patient', 'name', 'hemi', 'x', 'y', 'z', 'is_wm']],
        left_on=['patient_id', 'contact_name'],
        right_on=['patient', 'name'], how='left',
    )
    n_total   = len(merged)
    n_matched = int(merged['x'].notna().sum())
    n_missing = n_total - n_matched
    print(f'  rows: {n_total}  matched: {n_matched}  unmatched: {n_missing} ({100*n_missing/max(n_total,1):.1f}%)')

    if n_missing:
        unm = merged[merged['x'].isna()][['patient_id','condition','electrode','file_path','contact_name']].drop_duplicates()
        unm.to_csv(out_dir / 'UNMATCHED_contacts.csv', index=False)
    merged.to_csv(out_dir / f'{algo_tag}_{m["run_id"]}__with_fsaverage.csv', index=False)

    plot_df = merged.dropna(subset=['x','y','z']).copy()
    if not KEEP_WM and 'is_wm' in plot_df.columns:
        plot_df = plot_df[plot_df['is_wm'] == 0]
    if plot_df.empty:
        print('  [skip] no plottable rows'); continue

    clusters = sorted(int(c) for c in plot_df[cluster_col].dropna().unique())
    print(f'  clusters to plot: {len(clusters)}')

    for cl in clusters:
        sub = plot_df[plot_df[cluster_col] == cl]
        # Two-digit zero padding so the website (loadClusterRecon, padStart 2)
        # matches `cluster_07/...` and the like.
        cl_dir = out_dir / f'cluster_{cl:02d}'
        coords_mm = sub[['x','y','z']].to_numpy(dtype=float)

        # ---- by patient ----
        pats = sub['patient_id'].astype(str).tolist()
        pal_p, uniq_p = _palette_patients(pats)
        cols_p = [pal_p[v] for v in pats]
        brain = _new_brain()
        _add_spheres(brain, coords_mm, cols_p)
        bp_dir = cl_dir / 'by_patient'
        _save_views(brain, bp_dir)
        _save_legend(pal_p, uniq_p, 'patient', bp_dir / 'legend.png')
        brain.close()

        # ---- by condition ----
        conds = sub['condition'].astype(str).tolist()
        pal_c, uniq_c = _palette(conds, cmap_name='tab10')
        cols_c = [pal_c[v] for v in conds]
        brain = _new_brain()
        _add_spheres(brain, coords_mm, cols_c)
        bc_dir = cl_dir / 'by_condition'
        _save_views(brain, bc_dir)
        _save_legend(pal_c, uniq_c, 'condition', bc_dir / 'legend.png')
        brain.close()

        summary_rows.append({
            'method': m['method'], 'feature_set': m['feature_set'], 'run_id': m['run_id'],
            'cluster': cl,
            'n_contacts': len(sub),
            'n_patients': sub['patient_id'].nunique(),
            'n_conditions': sub['condition'].nunique(),
        })
        print(f'    cluster {cl:02d}: {len(sub)} contacts -> {cl_dir}')

    # Per-run recon summary (small JSON the website / future code can pick up)
    with open(out_dir / 'recon_summary.json', 'w') as f_out:
        json.dump({
            'method': m['method'], 'feature_set': m['feature_set'], 'run_id': m['run_id'],
            'modes': ['by_patient', 'by_condition'],
            'views': list(VIEWS.keys()),
            'clusters': clusters,
            'cluster_dir_format': 'cluster_{:02d}',
        }, f_out, indent=2)
    print(f'  done -> {out_dir}')

if summary_rows:
    summ = pd.DataFrame(summary_rows)
    summ_path = CLUSTERING_DIR / 'clustering_recon_summary.csv'
    summ.to_csv(summ_path, index=False)
    print(f'\nSaved summary ({len(summ)} cluster rows) -> {summ_path}')
else:
    print('\nNo summary rows produced.')



No summary rows produced.


In [8]:
# # === Debug: confirm L-vs-R hardcoding (and case sensitivity) cause unmatched contacts ===
# # Re-runs the join using BOTH the old (L-only, case-sensitive) rule and the new (L|R, case-insensitive) rule
# # and reports unmatched contacts per patient with samples + the FULL fsav contact list per patient.
# import re as _re

# _RX_OLD = _re.compile(r'_ERSP_([A-Za-z]+)_L(\d+)_TN', _re.IGNORECASE)
# _RX_NEW = _re.compile(r'_ERSP_([A-Za-z]+)_([LR])(\d+)_TN', _re.IGNORECASE)

# def _contact_old(electrode, file_path):
#     base = _basename_no_ext(file_path)
#     m = _RX_OLD.search(base)
#     if m:
#         return f'{m.group(1)}L{m.group(2)}'   # original: keeps original case
#     if isinstance(electrode, str) and electrode.strip():
#         return electrode.strip()
#     return None

# def _contact_new(electrode, file_path):
#     base = _basename_no_ext(file_path)
#     m = _RX_NEW.search(base)
#     if m:
#         return f'{m.group(1).upper()}{m.group(2).upper()}{m.group(3)}'   # uppercase normalized
#     if isinstance(electrode, str) and electrode.strip():
#         return electrode.strip().upper()
#     return None

# # Use the first clustering CSV for diagnosis
# _algo, _csv = clustering_csvs[0]
# df_dbg = pd.read_csv(_csv)
# df_dbg['contact_old'] = df_dbg.apply(lambda r: _contact_old(r['electrode'], r['file_path']), axis=1)
# df_dbg['contact_new'] = df_dbg.apply(lambda r: _contact_new(r['electrode'], r['file_path']), axis=1)

# # OLD rule: case-sensitive join against fsav as-is
# fsav_keys_old = set(zip(fsav['patient'].astype(str), fsav['name'].astype(str)))
# # NEW rule: case-insensitive join (uppercase both sides)
# fsav_keys_new = set(zip(fsav['patient'].astype(str), fsav['name'].astype(str).str.upper()))

# df_dbg['matched_old'] = [(str(p), str(c)) in fsav_keys_old for p, c in zip(df_dbg['patient_id'], df_dbg['contact_old'])]
# df_dbg['matched_new'] = [(str(p), str(c)) in fsav_keys_new for p, c in zip(df_dbg['patient_id'], df_dbg['contact_new'])]

# print(f'CSV: {_csv.name}')
# print(f'Total rows: {len(df_dbg)}')
# print(f'  matched (OLD rule, L only, case-sensitive)  : {df_dbg["matched_old"].sum()}  unmatched: {(~df_dbg["matched_old"]).sum()}')
# print(f'  matched (NEW rule, L|R, case-insensitive)   : {df_dbg["matched_new"].sum()}  unmatched: {(~df_dbg["matched_new"]).sum()}')

# print('\nPer-patient unmatched count under OLD rule (top 20):')
# print(df_dbg[~df_dbg['matched_old']].groupby('patient_id').size().sort_values(ascending=False).head(20))

# recovered = df_dbg[(~df_dbg['matched_old']) & df_dbg['matched_new']]
# print(f'\nRecovered by NEW rule: {len(recovered)} contacts')
# print('Sample recovered (patient, contact_old -> contact_new, basename):')
# for _, r in recovered.head(10).iterrows():
#     print(f"  {r['patient_id']:>8}  {str(r['contact_old']):<10} -> {str(r['contact_new']):<10}  {_basename_no_ext(r['file_path'])}")

# still = df_dbg[~df_dbg['matched_new']]
# print(f'\nStill unmatched after NEW rule: {len(still)} contacts')
# print('Sample still-unmatched (one row per patient + FULL fsav contact list for that patient):')
# import pprint as _pp
# for pid, grp in still.groupby('patient_id'):
#     r = grp.iloc[0]
#     fsav_for_p = sorted(fsav.loc[fsav['patient'].astype(str) == str(pid), 'name'].astype(str).str.upper().unique().tolist())
#     print(f"  {pid:>8}  contact_new={r['contact_new']!r:<14}  basename={_basename_no_ext(r['file_path'])}")
#     print(f"           fsav names ({len(fsav_for_p)}) for {pid}: {fsav_for_p}")